In [ ]:
# ==========================================
# BLOCK 1: SMART ENVIRONMENT SETUP
# ==========================================
# @title ⚙️ 1. Setup Environment & Persistence

DRIVE_WORKSPACE_NAME = "AutoScribe_Workspace" # @param {type:"string"}
SHARED_DRIVE_NAME = "" # @param {type:"string"}

import os, sys, shutil
from datetime import datetime
from google.colab import drive
from IPython.display import clear_output, display
from IPython.utils import capture
import ipywidgets as widgets

# --- WORKSPACE CONFIGURATION ---
LOCAL_WORKSPACE = "/content/AutoScribe_Local"
TEMP_DIR = f"{LOCAL_WORKSPACE}/temp_media"
LOCAL_OUTPUT = f"{LOCAL_WORKSPACE}/outputs"
LOCAL_LOG = f"{LOCAL_WORKSPACE}/autoscribe_debug.log"

os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

def log_event(level, message, print_to_console=True):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"[{timestamp}] {level}: {message}"
    with open(LOCAL_LOG, "a", encoding="utf-8") as f:
        f.write(log_entry + "\n")
    if print_to_console:
        color = {"INFO": "32", "WARNING": "33", "ERROR": "31"}.get(level, "37")
        print(f"\033[{color}m{log_entry}\033[0m", flush=True)

def inf(msg, style, wdth):
    display(widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth)))

# 1. Smart Drive Mounting
if not os.path.exists('/content/drive/MyDrive'):
    log_event("INFO", "Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
else:
    log_event("INFO", "✅ Google Drive already mounted. Skipping.")

# 2. Path Resolution
root_path = f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}" if SHARED_DRIVE_NAME and os.path.exists(f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}") else "/content/drive/MyDrive"
workspace_name = DRIVE_WORKSPACE_NAME.strip() or "AutoScribe_Workspace"
DRIVE_BASE = f"{root_path}/{workspace_name}"
DRIVE_OUTPUT_DIR = os.path.join(DRIVE_BASE, "outputs")
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# 3. Smart Dependency Check
try:
    import faster_whisper
    log_event("INFO", "✅ Dependencies already present. Skipping installation.")
except ImportError:
    log_event("INFO", "Installing dependencies to local NVMe...")
    with capture.capture_output() as cap:
        !pip install -q yt-dlp faster-whisper ffmpeg-python transformers accelerate torch torchvision
        !pip install -q --upgrade --no-deps yt-dlp

os.environ["HF_HOME"] = "/content/local_hf_cache"
BLOCK_1_COMPLETED = True
clear_output()
inf('\u2714 Environment Synchronized (Resilient Setup)', 'success', '400px')
log_event("INFO", "Block 1 Initialization Complete.")

In [ ]:
# ==========================================
# BLOCK 2: INGESTION & ROUTING
# ==========================================
# @title 📥 2. Source Configuration & Routing

# @markdown ### ⚙️ 1. Select Source
SOURCE_TYPE = "URL" # @param ["URL", "Google_Drive"]

# @markdown ---
# @markdown ### 🔗 Option A: URL
YOUTUBE_URL = "https://youtube.com/playlist?list=..." # @param {type:"string"}

# @markdown ### 📂 Option B: Google Drive
DRIVE_FILE_PATH = "/content/drive/MyDrive/your_video.mp4" # @param {type:"string"}
# @markdown ---

# @markdown ### 🧠 2. AI Processing Settings
WHISPER_MODE = "Auto" # @param ["On", "Off", "Auto"]
VISION_FALLBACK = True # @param {type:"boolean"}

if 'BLOCK_1_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Run Block 1 first.")

import yt_dlp
import subprocess
import shutil
import os

routing_queue = []

def log_event(level, message, print_to_console=True):
    """Internal logging helper for Block 2 (Logic mirrored from Block 1)"""
    if print_to_console: print(message, flush=True)

def analyze_and_route(file_path, title, video_id):
    log_event("INFO", f"Analyzing audio stream for: {title}...")
    command = ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=codec_type", "-of", "default=nw=1:nk=1", file_path]
    requires_vision = False
    
    try:
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        has_audio = "audio" in result.stdout.strip().lower()
        
        if WHISPER_MODE == "Auto":
            if not has_audio and VISION_FALLBACK:
                requires_vision = True
                log_event("INFO", f"[ROUTING]: No audio detected. Tagged for Vision analysis.")
            else:
                log_event("INFO", f"[ROUTING]: Audio intact. Tagged for Whisper transcription.")
        elif WHISPER_MODE == "Off" and VISION_FALLBACK:
            requires_vision = True
            log_event("INFO", f"[ROUTING]: Whisper forced OFF. Tagged for Vision analysis.")
            
    except Exception as e:
        log_event("ERROR", f"Audio analysis failed for {title}: {e}. Fallback to Vision.")
        requires_vision = True

    routing_queue.append({'id': video_id, 'title': title, 'path': file_path, 'use_vision': requires_vision})

log_event("INFO", f"Processing source type: {SOURCE_TYPE}")

# --- GOOGLE DRIVE LOGIC ---
if SOURCE_TYPE == "Google_Drive":
    if os.path.exists(DRIVE_FILE_PATH):
        filename = os.path.basename(DRIVE_FILE_PATH)
        dest_path = os.path.join(TEMP_DIR, filename)
        
        # Resilient Check: Don't copy if file already exists in local cache
        if not os.path.exists(dest_path):
            log_event("INFO", f"Copying {filename} to local NVMe...")
            shutil.copy2(DRIVE_FILE_PATH, dest_path)
        else:
            log_event("INFO", f"✅ {filename} already exists in local cache. Skipping copy.")
            
        analyze_and_route(dest_path, filename, filename.split('.')[0])
    else:
        log_event("ERROR", f"File not found on Drive: {DRIVE_FILE_PATH}")

# --- URL / YOUTUBE LOGIC ---
elif SOURCE_TYPE == "URL":
    # Android client spoofing to bypass bot detection
    common_extractor_args = {'youtube': ['player_client=android']}
    
    ydl_opts_meta = {
        'extract_flat': 'in_playlist', 
        'quiet': True,
        'extractor_args': common_extractor_args
    }
    
    with yt_dlp.YoutubeDL(ydl_opts_meta) as ydl:
        try:
            info = ydl.extract_info(YOUTUBE_URL, download=False)
            entries = info['entries'] if 'entries' in info else [info]
            
            for entry in entries:
                v_id = entry['id']
                title = entry.get('title', v_id)
                url = entry.get('url', YOUTUBE_URL)
                
                # Dynamic filename resolution to check for existing cache
                ydl_temp = yt_dlp.YoutubeDL({'format': 'bestaudio/best', 'outtmpl': f'{TEMP_DIR}/{v_id}.%(ext)s'})
                expected_filename = ydl_temp.prepare_filename(entry)
                
                if os.path.exists(expected_filename):
                    log_event("INFO", f"✅ {title} already cached locally. Skipping download.")
                    analyze_and_route(expected_filename, title, v_id)
                else:
                    log_event("INFO", f"Downloading: {title}")
                    dl_opts = {
                        'format': 'bestaudio/best',
                        'outtmpl': f'{TEMP_DIR}/{v_id}.%(ext)s',
                        'quiet': True,
                        'extractor_args': common_extractor_args
                    }
                    with yt_dlp.YoutubeDL(dl_opts) as ydl_dl:
                        dl_info = ydl_dl.extract_info(url, download=True)
                        local_file = ydl_dl.prepare_filename(dl_info)
                        analyze_and_route(local_file, title, v_id)
                        
        except Exception as e:
            log_event("ERROR", f"URL parsing failed: {e}")

BLOCK_2_COMPLETED = True
log_event("INFO", f"✅ Block 2 Complete. {len(routing_queue)} item(s) in queue.")

In [ ]:
# ==========================================
# BLOCK 3: ADAPTIVE AI PROCESSING
# ==========================================
# @title 🧠 3. Execute AI Processing
# @markdown ---
# @markdown ### 🤖 Adaptive Processing Engine
# @markdown This block performs the following automated steps:
# @markdown 1. **Duration Analysis:** Checks the length of each media file.
# @markdown 2. **Model Scaling:** Automatically selects `large-v3` for standard files or `medium` for ultra-long content (>4h) to ensure memory stability.
# @markdown 3. **VAD Filtering:** Activates Voice Activity Detection to clean audio and reduce RAM overhead.
# @markdown 4. **Resource Management:** Flushes GPU VRAM and System RAM between every task.
# @markdown 5. **Cloud Sync:** Securely exports all generated Markdown files to your Google Drive workspace.
# @markdown ---

import gc, torch, subprocess, os, re
from datetime import datetime
from PIL import Image

# Safety check for preceding blocks
if 'BLOCK_2_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Run Block 1 and 2 before Block 3.")

print("🚀 INITIALIZING BLOCK 3 - Adaptive AI Engine...", flush=True)

try:
    from faster_whisper import WhisperModel
    from google.colab import userdata
    # HF_TOKEN is required for gated models or specific API access
    hf_token = userdata.get('HF_TOKEN')
except Exception as e:
    print(f"⚠️ Note: HF_TOKEN retrieval skipped or not found. Using local cache.")

def get_duration(file_path):
    """Retrieves file duration in minutes using ffprobe."""
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", file_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, text=True)
    return float(result.stdout)/60 if result.stdout else 0

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
LOCAL_EXPORT_DIR = os.path.join(LOCAL_OUTPUT, f"Run_{run_timestamp}")
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

# --- WHISPER ADAPTIVE PROCESSING ---
audio_tasks = [t for t in routing_queue if not t['use_vision']]
if audio_tasks:
    for task in audio_tasks:
        duration_mins = get_duration(task['path'])
        # Adaptive logic: Downscale model if file is > 4 hours to prevent OOM
        model_size = "medium" if duration_mins > 240 else "large-v3"
        
        print(f"🎙️ Processing {task['title']} ({duration_mins:.1f}m) using {model_size} model...", flush=True)
        
        try:
            model = WhisperModel(model_size, device="cuda", compute_type="float16", download_root=os.environ["HF_HOME"])
            segments, _ = model.transcribe(
                task['path'], 
                language="sv", 
                vad_filter=True, 
                condition_on_previous_text=False
            )
            
            safe_name = re.sub(r'[^\w\s-]', '', task['title']).strip().replace(' ', '_')[:50]
            export_path = os.path.join(LOCAL_EXPORT_DIR, f"{safe_name}.md")
            
            with open(export_path, "w", encoding="utf-8") as f:
                f.write(f"# Source: {task['title']}\n")
                f.write(f"**Processing Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
                f.write(f"**Model Used:** {model_size} (Adaptive Scaling)\n\n")
                f.write("## Transcription\n\n")
                for s in segments:
                    f.write(f"[{int(s.start)//60:02d}:{int(s.start)%60:02d}] {s.text.strip()}\n")
            
            print(f"✅ Successfully transcribed: {task['title']}", flush=True)
            
        except Exception as e:
            print(f"❌ ERROR: Whisper failed for {task['title']}: {e}", flush=True)
        finally:
            # Aggressive memory release
            if 'model' in locals(): del model
            gc.collect()
            torch.cuda.empty_cache()

# --- VISION PROCESSING (MOONDREAM2) ---
vision_tasks = [t for t in routing_queue if t['use_vision']]
if vision_tasks:
    print("--- Loading Moondream2 Vision Model ---", flush=True)
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        model_id = "vikhyatk/moondream2"
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, cache_dir=os.environ["HF_HOME"])
        moondream = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, cache_dir=os.environ["HF_HOME"]).to("cuda")
        moondream.eval()

        for task in vision_tasks:
            print(f"👁️ Analyzing Visuals: {task['title']}", flush=True)
            safe_name = re.sub(r'[^\w\s-]', '', task['title']).strip().replace(' ', '_')[:50]
            export_path = os.path.join(LOCAL_EXPORT_DIR, f"{safe_name}_vision.md")
            
            frame_dir = os.path.join(TEMP_DIR, f"frames_{task['id']}")
            os.makedirs(frame_dir, exist_ok=True)
            subprocess.run(["ffmpeg", "-i", task['path'], "-vf", "fps=1/30", f"{frame_dir}/frame_%04d.jpg", "-hide_banner", "-loglevel", "error"])
            
            frames = sorted(os.listdir(frame_dir))
            with open(export_path, "w", encoding="utf-8") as f:
                f.write(f"# Source: {task['title']}\n**Analysis Type:** Visual Frame Analysis\n\n")
                for i, frame_file in enumerate(frames):
                    try:
                        img_path = os.path.join(frame_dir, frame_file)
                        enc_image = moondream.encode_image(Image.open(img_path))
                        answer = moondream.answer_question(enc_image, "Describe the key action or text in this frame concisely.", tokenizer)
                        f.write(f"[{divmod(i * 30, 60)[0]:02d}:{divmod(i * 30, 60)[1]:02d}] {answer}\n")
                    except Exception as e: 
                        print(f"⚠️ Warning: Frame error: {e}")
                    
    except Exception as e: 
        print(f"❌ ERROR: Vision Model failed: {e}")
    finally:
        if 'moondream' in locals(): del moondream
        gc.collect()
        torch.cuda.empty_cache()

# --- FINAL EXPORT ---
print("📤 Syncing local outputs to Google Drive...", flush=True)
DRIVE_FINAL_DIR = os.path.join(DRIVE_OUTPUT_DIR, f"AutoScribe_Run_{run_timestamp}")
shutil.copytree(LOCAL_EXPORT_DIR, DRIVE_FINAL_DIR)

BLOCK_3_COMPLETED = True
print(f"✅ Block 3 Complete. Output directory: {DRIVE_FINAL_DIR}")

In [ ]:
# ==========================================
# BLOCK 4: FINALIZE & DISCONNECT
# ==========================================
# @title 📝 4. Finalize & Disconnect

import shutil, os
from google.colab import runtime

# @markdown ---
# @markdown **Warning:** This will wipe the local NVMe cache and disconnect the session.
CONFIRM_SHUTDOWN = True # @param {type:"boolean"}

if BLOCK_3_COMPLETED and CONFIRM_SHUTDOWN:
    log_event("INFO", "🧹 Starting final cleanup...")
    
    # Backup log to Drive before wiping local storage
    if os.path.exists(LOCAL_LOG):
        shutil.copy2(LOCAL_LOG, os.path.join(DRIVE_OUTPUT_DIR, f"debug_{datetime.now().strftime('%H%M%S')}.log"))
    
    if os.path.exists(LOCAL_WORKSPACE):
        shutil.rmtree(LOCAL_WORKSPACE)
        print("✅ Local NVMe storage cleared.")
    
    inf('🎉 WORKFLOW COMPLETE - Session Terminating', 'info', '400px')
    runtime.unassign()
else:
    log_event("WARNING", "Shutdown aborted or Block 3 was not completed.")